### Imports

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

### Loading Data

In [ ]:
data_path = "datasets"
contract_path = os.path.join(data_path, "contract.csv")
internet_path = os.path.join(data_path, "internet.csv")
personal_path = os.path.join(data_path, "personal.csv")
phone_path = os.path.join(data_path, "phone.csv")

df_contract = pd.read_csv(contract_path)
df_internet = pd.read_csv(internet_path)
df_personal = pd.read_csv(personal_path)
df_phone = pd.read_csv(phone_path)

### Exploratory Data Analysis and Data Wrangling

#### contract dataframe

In [ ]:
df_contract.info()

In [ ]:
df_contract.sample(10)

Things to check in general in this dataframe.
- Check for duplicate rows in the dataframe

Things to check within this dataframe;

- custormerID: if there are nulls or repeated IDs
- BeginDate: if there are nulls or incorrect dates (greater than today or earlier than a reasonable past date). Change format to pd.Datetime
- EndDate: if there are nulls or incorrect dates (earlier than BeginDate or later than today); transform "No" into pd.NaT value
- Type: check uniques in order to transform into a categorical variable
- PaperlessBilling: check uniques in order to transform into a categorical variable
- PaymentMethod: check uniques in order to transform into a categorical variable
- MonthlyCharges: check for nulls and incorrect values (negative or excessively high)
- TotalCharges: check for nulls and incorrect values (negative or excessively high)


##### General

In [ ]:
# Check for nulls
df_contract.isnull().sum()

In [ ]:
# Check for duplicate rows in the contract dataframe
df_contract.duplicated().sum()

##### customerID

In [ ]:
# Check for duplicate customerID in the contract dataframe
sum(df_contract["customerID"].value_counts() > 1)

In [ ]:
# Check null or empty values in the contract dataframe


##### BeginDate

In [ ]:
# Convert BeginDate to datetime format
df_contract["BeginDate"] = pd.to_datetime(df_contract["BeginDate"], errors="coerce", format="%Y-%m-%d")
df_contract.sample(10)

##### EndDate

In [ ]:
# Convert EndDate to datetime format
df_contract["EndDate"] = pd.to_datetime(df_contract["EndDate"], errors="coerce", format="%Y-%m-%d %H:%M:%S")
df_contract.sample(10)

##### Type

In [ ]:
# Check unique types
df_contract["Type"].unique()

##### PaperlessBilling

In [ ]:
# Check unique types
df_contract["PaperlessBilling"].unique()

##### PaymentMethod

In [ ]:
# Check unique types
df_contract["PaymentMethod"].unique()

##### MonthlyCharges

In [ ]:
# check for negative or extreamly high values
min = df_contract[df_contract["MonthlyCharges"] < 0]
high = df_contract[df_contract["MonthlyCharges"] > 1000]
print("Negative MonthlyCharges:\n", min)
print("Extremely High MonthlyCharges:\n", high)

##### TotalCharges

In [ ]:
# This column is str; it will be changed to numerical
df_contract["TotalCharges"] = pd.to_numeric(df_contract["TotalCharges"], errors="coerce")

# check if there are nulls
df_contract[df_contract["TotalCharges"].isna()]
idx = df_contract[df_contract["TotalCharges"].isna()].index

As seen, the "TotalCharges" column had some non-numeric values which were converted to NaN.

This probably is due the end of the timelapse of the dataframe, where some entries have not accumulated any charges yet.

Therefore, it might be reasonable to fill these NaN values with the same monthly value, assuming no charges have been accumulated yet; but prio to do so, I'll explore which is the most recent date in the dataframe

In [ ]:
df_contract["BeginDate"].max()

In [ ]:
# Since my guess was correct: Replacing NaN values in the "TotalCharges" column with the value MonthlyCharges
df_contract["TotalCharges"] = df_contract["TotalCharges"].fillna(df_contract["MonthlyCharges"])

# Explore same rows after filling NaN values
df_contract.iloc[idx]

In [ ]:
# check for negative or extreamly high values
min = df_contract[df_contract["TotalCharges"] < 0]
high = df_contract[df_contract["TotalCharges"] > 5000]
print("Negative TotalCharges:\n", min)
print("Extremely High TotalCharges:\n", high)

#### internet dataframe

In [ ]:
df_internet.info()

In [ ]:
df_internet.sample(10)

Things to check in general in this dataframe.
- Check for duplicate rows in the dataframe

Things to check within this dataframe;

- custormerID: if there are nulls or repeated IDs
- InternetService: check uniques in order to transform into a categorical variable
- OnlineSecurity: check uniques in order to transform into a categorical variable
- OnlineBackup: check uniques in order to transform into a categorical variable
- TechSupport: check uniques in order to transform into a categorical variable
- StreamingTV: check uniques in order to transform into a categorical variable
- StreamingMovies: check uniques in order to transform into a categorical variable


##### General

In [ ]:
# Check for missing values
missings = df_internet.isna().sum()
# Check for duplicate rows
duplited = df_internet.duplicated().sum()
print("Missing values per column:\n", missings)
print("Duplicate rows in the dataframe: ", duplited)

##### InternetService

In [ ]:
df_internet['InternetService'].unique()

##### OnlineSecurity


In [ ]:
df_internet['OnlineSecurity'].unique()

##### OnlineBackup 


In [ ]:
df_internet['OnlineBackup'].unique()    

##### DeviceProtection

In [ ]:
df_internet['DeviceProtection'].unique()

##### TechSupport

In [ ]:
df_internet['TechSupport'].unique()

##### StreamingTV


In [ ]:
df_internet['StreamingTV'].unique()

##### StreamingMovies

In [ ]:
df_internet['StreamingMovies'].unique()

#### personal dataframe

In [ ]:
df_personal.info()

In [ ]:
df_personal.sample(10)

##### General

In [ ]:
# Missing Values
missings = df_personal.isna().sum()
# Duplicates
duplicates = df_personal.duplicated().sum()
print("Missing Values:\n", missings)
print("Duplicates:", duplicates)

##### gender

In [ ]:
df_personal["gender"].unique()

##### SeniorCitizen

In [ ]:
df_personal["SeniorCitizen"].unique()

##### Partner

In [ ]:
df_personal["Partner"].unique()

##### Dependents

In [ ]:
df_personal["Dependents"].unique()

#### phone dataframe

In [ ]:
df_phone.info()

In [ ]:
df_phone.sample(10)

##### General

In [ ]:
# Missing Values
missings = df_phone.isna().sum()
# Duplicates
duplicates = df_phone.duplicated().sum()
print("Missing Values:\n", missings)
print("Duplicates:", duplicates)

##### MultipleLines

In [ ]:
df_phone["MultipleLines"].unique()

#### Conclusions of EDA

- In general all dataframes are correct and do not include missing or null values.
- There are a few columns names in the dataframes that do not follow the PascalCase; this issue will be fixed when mergin dataframes
- The number of rows is different in each datataframe; contract has ; internet has ; personal has ; and phone has . This means that some customers have not contracted some services.
- The column SeniorCitizen in the dataframe personal

### Merge Dataframes and target creation

This shall be made using customerID as common (anchor) for all dataframes.
Strategy to fix NaN o missing values when merging: Since 

### Models Training

### Split data into training and testing sets

#### Functions for training and evaluation

#### Logistic Regression

#### Decision Tree

#### Random Forest

#### CatBoost

### Best Model Evaluation

In [ ]:
### Predictions

###